# Attention notes

(condensed from around 1:17:00 in the video https://www.youtube.com/watch?v=kCc8FmEb1nY)

This is a communication mechanism between indices in the embedding dimension.

It does not inherently know about position; this is why we need to encode it ourselves.

- Version 1: the most naive "let inputs in the n_embd dimension look back in the past" is taking the average of all past tokens. Literally add them up.
- Version 2: treat this as a matmul by building a lower triangular `n_embd * n_embd` matrix where each row `i=1..n_embd` looks like `[{1/i .. 1/i} {0 .. 0}]` and multiplying it out.
- Version 3: instead of the average, softmax across `[{i values ..} {-inf ..}]` to allow the weights to be learnable.

The formula is `Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V`

Where $d_k$ is the dimension of the keys (head size) to scale down variance to unit so that softmax works better without peaking too much.

### How might you derive the scaling issue?

On random inputs (seed 1337) at init without scaling, notice the peaking in softmaxed weights:
```
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8859, 0.1141, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0559, 0.5318, 0.4123, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3182, 0.1012, 0.4600, 0.1206, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0407, 0.2013, 0.1081, 0.2830, 0.3669, 0.0000, 0.0000, 0.0000],
        [0.4086, 0.0516, 0.0833, 0.0991, 0.2512, 0.1061, 0.0000, 0.0000],
        [0.0575, 0.0180, 0.0566, 0.0545, 0.1110, 0.6008, 0.1016, 0.0000],
        [0.3151, 0.0758, 0.1093, 0.0408, 0.1806, 0.0490, 0.0477, 0.1816]],
```

Can back into it by understanding how this happens from variance of adding `head_size` number of random variables. With scaling, you get the much more even init:
```
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6253, 0.3747, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2270, 0.3988, 0.3742, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2753, 0.2068, 0.3019, 0.2160, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1404, 0.2093, 0.1792, 0.2279, 0.2432, 0.0000, 0.0000, 0.0000],
        [0.2189, 0.1305, 0.1471, 0.1536, 0.1938, 0.1562, 0.0000, 0.0000],
        [0.1272, 0.0951, 0.1267, 0.1255, 0.1500, 0.2287, 0.1467, 0.0000],
        [0.1650, 0.1156, 0.1266, 0.0990, 0.1436, 0.1036, 0.1029, 0.1438]],
```


In [52]:
import torch
from torch.nn import functional as F

torch.manual_seed(1337)

# B (batch), T (time), C (channels)
B, T, C = 4, 8, 32

x = torch.randn((B, T, C))
head_size = 16
query = torch.nn.Linear(C, head_size)
key = torch.nn.Linear(C, head_size)
value = torch.nn.Linear(C, head_size)

# [B, T, C] -> [B, T, head_size]
q: torch.Tensor = query(x)
k: torch.Tensor = key(x)
v: torch.Tensor = value(x)

# [B, T, head_size] * [B, head_size, T] = [B, T, T]
wei = q @ k.transpose(-2, -1) / (head_size ** 0.5)
# Masked-fill lower-triangular with -inf
mask = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(mask == 0, float("-inf"))
# Softmax per-row, ie the final dimension.
wei = F.softmax(wei, dim=-1)
#print(mask)
print(wei[0])

# [B, T, T] * [B, T, head_size] = [B, T, head_size]
out = wei @ v




tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6253, 0.3747, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2270, 0.3988, 0.3742, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2753, 0.2068, 0.3019, 0.2160, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1404, 0.2093, 0.1792, 0.2279, 0.2432, 0.0000, 0.0000, 0.0000],
        [0.2189, 0.1305, 0.1471, 0.1536, 0.1938, 0.1562, 0.0000, 0.0000],
        [0.1272, 0.0951, 0.1267, 0.1255, 0.1500, 0.2287, 0.1467, 0.0000],
        [0.1650, 0.1156, 0.1266, 0.0990, 0.1436, 0.1036, 0.1029, 0.1438]],
       grad_fn=<SelectBackward0>)


In [ ]:
q = torch.randn((B, T, head_size))
k = torch.randn((B, T, head_size))
wei = q @ k.transpose(-2, -1) / (head_size ** 0.5)
print(q.var())
print(k.var())
print(wei.var())


tensor(0.9495)
tensor(1.0391)
tensor(0.8080)
tensor([[[ 1.7515e+00,  5.7440e-01,  3.5575e-01, -1.6358e-01,  1.1725e+00,
           1.4082e+00,  8.5492e-01,  2.4078e-01],
         [ 1.0225e+00, -1.0886e+00, -6.5273e-01, -1.4535e+00,  8.8378e-01,
          -2.3904e-01,  1.1255e-01,  1.1441e+00],
         [ 4.9897e-02,  1.8399e-01,  1.8064e-01, -1.7481e-01,  1.7949e-01,
          -1.9351e-01,  2.6377e-01, -4.4248e-02],
         [-2.4847e+00,  5.3513e-01, -3.0705e-01,  6.0362e-02,  3.0864e-01,
           1.0139e+00, -9.6733e-01,  2.1596e-01],
         [-5.9132e-01, -1.4213e-01,  2.4994e-01, -1.2423e+00, -4.3622e-01,
          -5.4629e-01, -3.8858e-01,  8.4310e-01],
         [ 1.5934e-01,  7.6284e-01,  3.4141e-02,  7.7016e-01,  3.1757e-01,
           1.0866e+00, -2.8358e-01,  1.7119e+00],
         [ 4.3111e-01,  2.3614e-01,  2.1870e-01, -3.6091e-01, -6.3731e-01,
          -3.6113e-01, -8.3063e-01,  9.8982e-01],
         [ 1.4408e+00, -1.2091e+00,  4.4005e-01, -1.1951e-01, -3.6490e-01,
     